In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from pandas import DataFrame
from typing import Optional, Union
import talib.abstract as ta
from technical import qtpylib

import ccxt
class Strategy:
    Periods = 10
    src = 'hl2' #from Pine script (high+low)/2
    Multiplier = 3.0
    changeATR = True
    showsignals = True
    highlighting = True
    def populate_indicators(self, dataframe: DataFrame):
            atr2 = ta.SMA(dataframe, timeperiod=self.Periods)

            if self.changeATR:
                dataframe['atr'] = ta.ATR(dataframe['high'], dataframe['low'], dataframe['close'], timeperiod=self.Periods)
            else:
                dataframe['atr'] = atr2

            if self.src == 'hl2':
                up = ((dataframe['high'] + dataframe['low']) / 2) - (self.Multiplier * dataframe['atr'])
                dn = ((dataframe['high'] + dataframe['low']) / 2) + (self.Multiplier * dataframe['atr'])
            else:
                up = dataframe['close'] - self.Multiplier * dataframe['atr']
                dn = dataframe['close'] + self.Multiplier * dataframe['atr']

            dataframe['up1'] = up.shift(1)
            dataframe['up1'] = dataframe['up1'].fillna(up)
            dataframe['up'] = np.where(dataframe['close'].shift(1) > dataframe['up1'],
                                    np.maximum(up, dataframe['up1']),
                                    up)

            dataframe['dn1'] = dn.shift(1)
            dataframe['dn1'] = dataframe['dn1'].fillna(dn)
            dataframe['dn'] = np.where(dataframe['close'].shift(1) < dataframe['dn1'],
                                    np.minimum(dn, dataframe['dn1']),
                                    dn)

            dataframe['trend'] = 1
            dataframe['trend'] = dataframe['trend'].shift(1)
            dataframe['trend'].fillna(1, inplace=True)
            
            dataframe['trend'] = np.where(
                (dataframe['trend'] == -1) & (dataframe['close'] > dataframe['dn1']), 1,
                np.where(
                    (dataframe['trend'] == 1) & (dataframe['close'] < dataframe['up1']), -1,
                    dataframe['trend']
                )
            )
            trend_shifted = dataframe['trend'].shift(1)

            dataframe['buySignal'] = (dataframe['trend'] == 1) & (trend_shifted == -1)
            dataframe['sellSignal'] = (dataframe['trend'] == -1) & (trend_shifted == 1)

            return dataframe

BTC_USD = ccxt.binance()
dataframe = BTC_USD.fetch_ohlcv('BTC/USDT', timeframe='15m', limit=1000)

dataframe_source = DataFrame(dataframe, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])

cl = Strategy()



/var/folders/y4/_hfv_1c12c5dst33jlmwh_g80000gn/T/ipykernel_91122/607164152.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe['atr'] = ta.ATR(dataframe['high'], dataframe['low'], dataframe['close'], timeperiod=self.Periods)
/var/folders/y4/_hfv_1c12c5dst33jlmwh_g80000gn/T/ipykernel_91122/607164152.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe['up1'] = up.shift(1)
/var/folders/y4/_hfv_1c12c5dst33jlmwh_g80000gn/T/ipykernel_91122/607164152.py:33: SettingWithCopyWarning: 
A value is

AttributeError: 'DataFrame' object has no attribute 'append'